# Gold Layer: Star Schema & Data Warehouse
**Mục tiêu:** Xây dựng mô hình **Galaxy Schema** (Nhiều Fact tables chia sẻ Dimension).
- **Dimensions:** Customers (SCD Type 2), Products, Sellers, Date.
- **Facts:** Sales (Grain: Item), Payments (Grain: Transaction).

In [0]:
from delta.tables import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

# --- CẤU HÌNH ---
catalog_name = "brazilian_ecommerce"
silver_schema = "silver"
gold_schema = "gold"

# Đường dẫn Checkpoint (Quan trọng cho Streaming)
checkpoint_base = "/Volumes/brazilian_ecommerce/gold/checkpoints/"

# Tạo Schema Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

print(f"✅ Source: {catalog_name}.{silver_schema}")
print(f"✅ Target: {catalog_name}.{gold_schema}")

In [0]:
def create_dim_date(start='2016-01-01', end='2020-12-31'):
    # Tạo dãy ngày liên tục
    df_date = spark.sql(f"""
        SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) as date_key
    """)
    
    # Tính toán các thuộc tính thời gian
    df_dim_date = df_date.select(
        col("date_key"),
        date_format(col("date_key"), "yyyyMMdd").cast("int").alias("date_id"),
        year(col("date_key")).alias("year"),
        quarter(col("date_key")).alias("quarter"),
        month(col("date_key")).alias("month"),
        weekofyear(col("date_key")).alias("week_of_year"),
        dayofweek(col("date_key")).alias("day_of_week"),
        date_format(col("date_key"), "EEEE").alias("day_name"),
        date_format(col("date_key"), "MMMM").alias("month_name"),
        when(dayofweek(col("date_key")).isin(1, 7), True).otherwise(False).alias("is_weekend")
    )
    
    # Ghi đè (Overwrite) vì bảng này tĩnh
    df_dim_date.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog_name}.{gold_schema}.dim_date")
    
    print("✅ dim_date created successfully.")

create_dim_date()

In [0]:
# 1. DIM CUSTOMERS (SCD Type 2 Structure)
df_cust_silver = spark.read.table(f"{catalog_name}.{silver_schema}.customers")
df_dim_customers = df_cust_silver \
    .withColumn("customer_sk", sha2(concat(col("customer_unique_id"), col("customer_city")), 256)) \
    .select(
        col("customer_sk"),
        col("customer_unique_id").alias("customer_natural_key"),
        col("customer_zip_code_prefix"),
        col("customer_city"),
        col("customer_state"),
        lit(True).alias("is_current"), # Cờ hiệu SCD
        current_date().alias("effective_start_date"),
        lit(None).cast("date").alias("effective_end_date")
    )
df_dim_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_customers")

# 2. DIM PRODUCTS
df_prod_silver = spark.read.table(f"{catalog_name}.{silver_schema}.products")
df_dim_products = df_prod_silver \
    .withColumn("product_sk", sha2(col("product_id"), 256)) \
    .select(
        col("product_sk"),
        col("product_id").alias("product_natural_key"),
        col("product_category_name"),
        col("category_english"),
        col("product_photos_qty"),
        col("product_weight_g")
    )
df_dim_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_products")

# 3. DIM SELLERS
df_sell_silver = spark.read.table(f"{catalog_name}.{silver_schema}.sellers")
df_dim_sellers = df_sell_silver \
    .withColumn("seller_sk", sha2(col("seller_id"), 256)) \
    .select(
        col("seller_sk"),
        col("seller_id").alias("seller_natural_key"),
        col("seller_city"),
        col("seller_state")
    )
df_dim_sellers.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog_name}.{gold_schema}.dim_sellers")

print("✅ All Dimensions Created.")

In [0]:
# Xóa bảng và Checkpoint để chạy lại từ đầu
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{gold_schema}.fact_sales")
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.{gold_schema}.fact_payments")

# Xóa file checkpoint rác
dbutils.fs.rm(f"{checkpoint_base}/fact_sales", True)
dbutils.fs.rm(f"{checkpoint_base}/fact_payments", True)

print("✅ Đã dọn dẹp sạch sẽ. Sẵn sàng chạy lại code mới.")

In [0]:
# --- FACT SALES (ĐÃ FIX LỖI KEY) ---

def upsert_fact_sales(batch_df, batch_id):
    # 1. Load Dimensions
    # (Dùng filter is_current=true để lấy bản ghi mới nhất)
    dim_cust = spark.read.table(f"{catalog_name}.{gold_schema}.dim_customers") \
        .filter("is_current = true") \
        .select("customer_natural_key", "customer_sk")
        
    dim_prod = spark.read.table(f"{catalog_name}.{gold_schema}.dim_products").select("product_natural_key", "product_sk")
    dim_sell = spark.read.table(f"{catalog_name}.{gold_schema}.dim_sellers").select("seller_natural_key", "seller_sk")
    
    # 2. LOAD BRIDGE: Đọc bảng Silver Customer để lấy map: customer_id -> unique_id
    cust_bridge = spark.read.table(f"{catalog_name}.{silver_schema}.customers") \
        .select("customer_id", "customer_unique_id")
    
    # 3. JOIN LOGIC
    # Bước A: Join Orders với Bridge để lấy Unique ID
    batch_with_uid = batch_df.join(cust_bridge, "customer_id", "left")
    
    # Bước B: Join với Dim bằng Unique ID (customer_natural_key)
    enriched_df = batch_with_uid \
        .join(dim_cust, batch_with_uid.customer_unique_id == dim_cust.customer_natural_key, "left") \
        .join(dim_prod, batch_with_uid.product_id == dim_prod.product_natural_key, "left") \
        .join(dim_sell, batch_with_uid.seller_id == dim_sell.seller_natural_key, "left") \
        .select(
            col("order_id"),
            col("order_item_id"),
            col("customer_sk"), # Giờ đây cột này sẽ có dữ liệu chuẩn
            col("product_sk"),
            col("seller_sk"),
            col("order_purchase_timestamp"),
            date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("date_id"),
            col("price").alias("sales_amount"),
            col("freight_value"),
            col("order_status")
        )
    
    # 4. Write Append
    enriched_df.write.format("delta").mode("append").saveAsTable(f"{catalog_name}.{gold_schema}.fact_sales")

# --- EXECUTE STREAM ---
print("⏳ Processing Fact Sales (Fixing Keys)...")
static_orders = spark.read.table(f"{catalog_name}.{silver_schema}.orders")

query_sales = (spark.readStream.table(f"{catalog_name}.{silver_schema}.order_items")
              .join(static_orders, "order_id", "inner") 
              .writeStream
              .foreachBatch(upsert_fact_sales)
              .option("checkpointLocation", f"{checkpoint_base}/fact_sales")
              .trigger(availableNow=True)
              .start())

query_sales.awaitTermination()
print("✅ Fact Sales Updated (Fixed).")

In [0]:
# --- FACT PAYMENTS (ĐÃ FIX LỖI KEY) ---

def upsert_fact_payments(batch_df, batch_id):
    # 1. Load Dim & Bridge
    dim_cust = spark.read.table(f"{catalog_name}.{gold_schema}.dim_customers") \
        .filter("is_current = true") \
        .select("customer_natural_key", "customer_sk")
        
    cust_bridge = spark.read.table(f"{catalog_name}.{silver_schema}.customers") \
        .select("customer_id", "customer_unique_id")
    
    # 2. Load Static Orders
    orders_df = spark.read.table(f"{catalog_name}.{silver_schema}.orders") \
        .select("order_id", "customer_id", "order_purchase_timestamp")
    
    # 3. Join Logic
    # Payment -> Orders -> Bridge (lấy Unique ID) -> Dim Customer (lấy SK)
    pay_with_order = batch_df.join(orders_df, "order_id", "inner")
    pay_with_uid = pay_with_order.join(cust_bridge, "customer_id", "left")
    
    final_df = pay_with_uid \
        .join(dim_cust, pay_with_uid.customer_unique_id == dim_cust.customer_natural_key, "left") \
        .select(
            col("order_id"),
            col("payment_sequential"),
            col("customer_sk"),
            col("order_purchase_timestamp"),
            date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("date_id"),
            col("payment_type"),
            col("payment_installments"),
            col("payment_value")
        )
    
    # 4. Write Append
    final_df.write.format("delta").mode("append").saveAsTable(f"{catalog_name}.{gold_schema}.fact_payments")

# --- EXECUTE STREAM ---
print("⏳ Processing Fact Payments (Fixing Keys)...")
query_pay = (spark.readStream.table(f"{catalog_name}.{silver_schema}.order_payments")
             .writeStream
             .foreachBatch(upsert_fact_payments)
             .option("checkpointLocation", f"{checkpoint_base}/fact_payments")
             .trigger(availableNow=True)
             .start())

query_pay.awaitTermination()
print("✅ Fact Payments Updated (Fixed).")

In [0]:
# Kiểm tra nhanh số liệu
print("📊 --- GOLD LAYER REPORT ---")

count_sales = spark.read.table(f"{catalog_name}.{gold_schema}.fact_sales").count()
sum_sales = spark.read.table(f"{catalog_name}.{gold_schema}.fact_sales").agg(sum("sales_amount")).collect()[0][0]

count_pay = spark.read.table(f"{catalog_name}.{gold_schema}.fact_payments").count()
sum_pay = spark.read.table(f"{catalog_name}.{gold_schema}.fact_payments").agg(sum("payment_value")).collect()[0][0]

print(f"Fact Sales Rows: {count_sales} | Total Sales Revenue: {sum_sales:,.2f}")
print(f"Fact Payments Rows: {count_pay} | Total Payment Value: {sum_pay:,.2f}")